In [ ]:
# ============================================================
# CÀI ĐẶT THƯ VIỆN
# ============================================================
!pip install -q catboost xgboost lightgbm scikit-learn pandas numpy tqdm
!pip install -q transformers underthesea torch
print('✅ Đã cài xong thư viện')

✅ Đã cài xong thư viện


In [ ]:
import os
import sys
import subprocess
import time
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

def run_external_script(script_path):
    print(f"\n{'='*80}")
    print(f" ĐANG CHẠY SCRIPT: {script_path}")
    print(f"{'='*80}")
    start_time = time.time()
    try:
        subprocess.run([sys.executable, script_path], check=True)
        print(f" Hoàn thành {script_path} trong {time.time() - start_time:.1f}s")
    except subprocess.CalledProcessError as e:
        print(f"Lỗi khi chạy {script_path}. Mã lỗi: {e.returncode}")
        sys.exit(1)

def main():
    # ---------------------------------------------------------------------
    # BƯỚC 1 & 2: FEATURE ENGINEERING
    # ---------------------------------------------------------------------
    # Chạy hai file fe/upgrade_part1_cleaning.py và fe/upgrade_part2_features.py các bước này phải chạy phoBert khá lâu nên lấy tập train_fe_v2.csv và test_fe_v2.csv đã được chạy sẵn 

    # Load dữ liệu FE
    train = pd.read_csv('data/train_fe_v2.csv')
    test = pd.read_csv('data/test_fe_v2.csv')

    # =====================================================================
    # A. HUẤN LUYỆN CATBOOST
    # =====================================================================
    print("[1/4] Đang huấn luyện CatBoost (Lấy 70% trọng số)...")
    X_num = train.select_dtypes(include=['number']).drop(columns=['Academic_Status', 'Student_ID'], errors='ignore')
    y_num = train['Academic_Status'].astype(int)
    X_test_num = test.select_dtypes(include=['number']).drop(columns=['Student_ID'], errors='ignore')[X_num.columns]

    class_weights_cat = [1.0, 1.8, 2.5]
    skf_cat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_probs_cat = np.zeros((len(test), 3))

    for tr_idx, val_idx in skf_cat.split(X_num, y_num):
        X_tr, X_val = X_num.iloc[tr_idx], X_num.iloc[val_idx]
        y_tr, y_val = y_num.iloc[tr_idx], y_num.iloc[val_idx]

        model_cat = CatBoostClassifier(
            iterations=1000, learning_rate=0.03, depth=7,
            class_weights=class_weights_cat, loss_function='MultiClass',
            eval_metric='TotalF1', task_type="GPU", verbose=0,
            early_stopping_rounds=100,
            random_state=10
        )
        model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        test_probs_cat += model_cat.predict_proba(X_test_num) / 5


    # =====================================================================
    # B. HUẤN LUYỆN XGBOOST
    # =====================================================================
    print("[2/4] Đang huấn luyện XGBoost với Impact Features (Lấy 30% trọng số)...")
    def add_impact_features(df):
        df = df.copy()
        if 'Has_Debt' in df.columns and 'Mean_Score_All' in df.columns:
            df['debt_score_ratio'] = df['Has_Debt'] * (10 - df['Mean_Score_All'])
        if 'Personal_Essay' in df.columns:
            df['essay_len'] = df['Personal_Essay'].str.len().fillna(0)
        return df

    train_xgb = add_impact_features(train)
    test_xgb = add_impact_features(test)

    X_xgb = train_xgb.select_dtypes(include=['number']).drop(columns=['Academic_Status', 'Student_ID'], errors='ignore')
    y_xgb = train_xgb['Academic_Status'].astype(int)
    X_test_xgb = test_xgb.select_dtypes(include=['number']).drop(columns=['Student_ID'], errors='ignore')[X_xgb.columns]

    model_xgb = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.015, max_depth=5,
        subsample=0.7, colsample_bytree=0.7, tree_method='hist', device='cuda',
        random_state=42
    )
    model_xgb.fit(X_xgb, y_xgb)
    test_probs_xgb = model_xgb.predict_proba(X_test_xgb)


    # =====================================================================
    # C. TRỘN HAI MÔ HÌNH VÀ TẠO BẢN BASELINE
    # =====================================================================
    w_xgb = 0.30
    w_cat = 0.70
    t_dropout = 1.15

    final_base_probs = (test_probs_xgb * w_xgb) + (test_probs_cat * w_cat)
    final_base_probs[:, 2] *= t_dropout
    base_preds = np.argmax(final_base_probs, axis=1)

    base_df = pd.DataFrame({'Student_ID': test['Student_ID'], 'Academic_Status': base_preds})
    # =====================================================================
    # D. ENSEMBLE VÀ CHỌN LỌC DỰA TRÊN XÁC SUẤT
    # =====================================================================
    print("\n [3/4] Chạy Ensemble 5-Fold (Trích xuất độ tin cậy ELITE)...")

    # Chuẩn bị dữ liệu full (bao gồm cả Categorical) cho Elite Ensemble
    X_full = train.drop(columns=['Academic_Status', 'Student_ID'], errors='ignore')
    y_full = train['Academic_Status'].astype(int)
    X_test_full = test[X_full.columns].copy()

    cat_features = X_full.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in cat_features:
        X_full[col] = X_full[col].astype(str).replace(['nan', 'NaN', 'None'], 'Unknown')
        X_test_full[col] = X_test_full[col].astype(str).replace(['nan', 'NaN', 'None'], 'Unknown')

    num_features = X_full.select_dtypes(include=['number']).columns.tolist()
    X_full[num_features] = X_full[num_features].fillna(0)
    X_test_full[num_features] = X_test_full[num_features].fillna(0)

    skf_elite = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_probs_elite = np.zeros((len(X_test_full), 3))
    elite_weights = {0: 0.9, 1: 1.8, 2: 3.8}

    for fold, (train_idx, val_idx) in enumerate(skf_elite.split(X_full, y_full)):
        X_tr, y_tr = X_full.iloc[train_idx], y_full.iloc[train_idx]

        model_elite = CatBoostClassifier(
            iterations=1000, learning_rate=0.03, depth=6,
            class_weights=elite_weights, cat_features=cat_features,
            task_type="GPU", verbose=0
        )
        model_elite.fit(X_tr, y_tr)
        test_probs_elite += model_elite.predict_proba(X_test_full) / 5
        print(f"  ✓ Đã xong Fold {fold+1}/5")

    print("\n [4/4] Kích hoạt ELITE FILTERING (Lọc > 80%)...")
    elite_df = base_df.copy()
    confidence_threshold = 0.80
    changed_count = 0

    for i in range(len(elite_df)):
        new_probs = test_probs_elite[i]
        new_label = np.argmax(new_probs)
        confidence = np.max(new_probs)
        old_label = base_df.at[i, 'Academic_Status']

        if old_label != new_label and confidence > confidence_threshold:
            elite_df.at[i, 'Academic_Status'] = new_label
            changed_count += 1

    output_file = 'submission_FINAL.csv'
    elite_df.to_csv(output_file, index=False)

    print("\n" + "="*80)
    print(f"HOÀN THÀNH PIPELINE TỪ A-Z!")
    print(f"Số lượng thay đổi (Elite corrections): {changed_count}")
    print(f"Đã xuất file: {output_file}")
    print(f"Phân bổ nhãn cuối: {elite_df['Academic_Status'].value_counts().to_dict()}")
    print("="*80)

if __name__ == "__main__":
    main()

[1/4] Đang huấn luyện CatBoost (Lấy 70% trọng số)...
[2/4] Đang huấn luyện XGBoost với Impact Features (Lấy 30% trọng số)...

 [3/4] Chạy Ensemble 5-Fold (Trích xuất độ tin cậy ELITE)...
  ✓ Đã xong Fold 1/5
  ✓ Đã xong Fold 2/5
  ✓ Đã xong Fold 3/5
  ✓ Đã xong Fold 4/5
  ✓ Đã xong Fold 5/5

 [4/4] Kích hoạt ELITE FILTERING (Lọc > 80%)...

HOÀN THÀNH PIPELINE TỪ A-Z!
Số lượng thay đổi (Elite corrections): 0
Đã xuất file: submission_FINAL.csv
Phân bổ nhãn cuối: {0: 2603, 1: 824, 2: 573}
